## 1. Libraries
`importlib.reload` makes edits in `strava_data/` pick up without restarting the kernel.

In [1]:
import importlib
import logging
import pandas as pd
import numpy as np

import strava_data
import strava_data.authentication
import strava_data.visualization
import strava_data.activity_cache
importlib.reload(strava_data)
importlib.reload(strava_data.authentication)
importlib.reload(strava_data.visualization)
importlib.reload(strava_data.activity_cache)

from strava_data.authentication import login
import strava_data.visualization as vis
# Disk-cached per-activity fetches keyed by id — only new activities hit the API,
# keeping re-runs well under Strava's short-term rate limit. Delete .cache/*.json to refresh.
from strava_data.activity_cache import fetch_text_fields, fetch_velocity_streams

# Show plots inline in the notebook (update_plots.py sets this to False for CI)
vis.SHOW_PLOTS = True

# Silence stravalib's per-request 'No rates present in response headers' warning
logging.getLogger('stravalib.util.limiter').setLevel(logging.ERROR)

In [2]:
# !pip install plotly ipywidgets anywidget nbformat

## 2. Login
Uses `secrets/client_secrets.txt` + `secrets/strava_token.json` locally, or `STRAVA_CLIENT_*` env vars in CI.

In [3]:
client = login(secrets_folder="../secrets")

You have already authenticated once before. Refreshing your token now.
Hi Joey, authentication successful!


In [4]:
activities_object = client.get_activities(limit=1000)
activities = list(activities_object)

In [17]:
activities[i].distance

9002.8

In [21]:
for i in range(30):
    if activities[i].type == 'Run':
        print(f"Activity {i}: {activities[i].name} ({activities[i].elapsed_time / 60:.1f} min, {activities[i].distance / 1000:.2f} km, {(activities[i].elapsed_time / 60) / (activities[i].distance / 1000):.2f} min/km")

Activity 1: Namiddagloop (46.2 min, 7.32 km, 6.31 min/km
Activity 2: Middagloop (24.3 min, 3.95 km, 6.14 min/km
Activity 3: Ochtendloop (48.1 min, 7.58 km, 6.35 min/km
Activity 13: Nachtloop (31.6 min, 4.05 km, 7.79 min/km
Activity 15: Eerste mini brick training 🧱 (34.2 min, 5.21 km, 6.56 min/km
Activity 18: Nuij sjeun poik (30.7 min, 5.85 km, 5.25 min/km
Activity 19: Völs/Preuswald trailke mit de boys (317.1 min, 25.76 km, 12.31 min/km
Activity 24: Middagloop (157.1 min, 21.90 km, 7.17 min/km
Activity 27: Trail run / hike (109.8 min, 11.38 km, 9.64 min/km
Activity 29: Nachtloop 🌧️ (56.5 min, 9.00 km, 6.28 min/km


In [30]:
# ============================================================
# INTERACTIVE STRAVA AEROBIC DECOUPLING ANALYZER
#
#   ┌───────────────────────────────────────────────────────┐
#   │ Aerobic Decoupling in: <session name>                 │
#   ├──────────────────────────────┬────────────────────────┤
#   │ Interval slider (4 handles)  │                        │
#   │ Pace / HR / Elevation        │  Statistics            │
#   │ GPS map                      │  (fixed width)         │
#   └──────────────────────────────┴────────────────────────┘
#
# Change `dashboard_width` to resize everything: the right
# panel keeps its width, the graphs and map take the rest.
# ============================================================

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import display, HTML

from strava_data.visualization import COLORS, STYLE


# ============================================================
# COLORS / GLOBAL STYLE
# ============================================================

BACKGROUND = "#000000"
BORDER = "#000000"
ORANGE = COLORS["main"]
WHITE = COLORS["neutral"]
GRAY = COLORS["dark"]
DARK_GRAY = COLORS["darker"]
GRID_COLOR = COLORS["darker"]

# One font stack for HTML, widgets and Plotly, so the browser picks the
# same fallback everywhere when the primary font is not installed.
FONT_FAMILY = f"{STYLE['font_family']}, Arial, sans-serif"

# Shared horizontal margins so the map lines up with the graph area.
MARGIN_LEFT = 95
MARGIN_RIGHT = 20


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def _format_pace(pace):
    """Convert decimal minutes/km to mm:ss/km."""
    if not np.isfinite(pace):
        return "—"
    total_seconds = pace * 60.0
    minutes = int(total_seconds // 60)
    seconds = int(round(total_seconds % 60))
    if seconds == 60:
        minutes += 1
        seconds = 0
    return f"{minutes}:{seconds:02d}"


def _format_duration(minutes):
    """Convert decimal minutes to mm:ss or hh:mm:ss."""
    if not np.isfinite(minutes):
        return "—"
    total_seconds = int(round(minutes * 60))
    hours = total_seconds // 3600
    mins = (total_seconds % 3600) // 60
    secs = total_seconds % 60
    if hours > 0:
        return f"{hours}:{mins:02d}:{secs:02d}"
    return f"{mins}:{secs:02d}"


def _format_change(value):
    """Format percentage change."""
    if not np.isfinite(value):
        return "—"
    return f"{value:+.2f}%"


def _safe_nanmean(values):
    """np.nanmean without warnings if all values are NaN."""
    values = np.asarray(values)
    if not np.any(np.isfinite(values)):
        return np.nan
    return np.nanmean(values)


def _safe_value(value, decimals=1):
    """Format numerical values while handling NaN."""
    if not np.isfinite(value):
        return "—"
    return f"{value:.{decimals}f}"


def _px(value):
    """Return a pixel width for ints or "1300px" strings, else None."""
    if isinstance(value, (int, float)):
        return int(value)
    if isinstance(value, str) and value.strip().endswith("px"):
        return int(float(value.strip()[:-2]))
    return None


def _route_zoom(latitude, longitude, width_px, height_px, padding=0.9):
    """
    Largest web-mercator zoom level at which the whole route fits in
    a map of width_px x height_px (MapLibre uses 512 px tiles).
    """
    lat = np.radians(latitude[np.isfinite(latitude)])
    lon = longitude[np.isfinite(longitude)]

    lon_span = max(float(np.ptp(lon)), 1e-6)
    merc_y = np.log(np.tan(np.pi / 4 + lat / 2))
    y_span = max(float(np.ptp(merc_y)), 1e-6)

    zoom_lon = np.log2(360.0 * width_px * padding / (512.0 * lon_span))
    zoom_lat = np.log2(2 * np.pi * height_px * padding / (512.0 * y_span))

    return float(np.clip(min(zoom_lon, zoom_lat), 1, 18))


def _percent_change(old, new):
    if np.isfinite(old) and np.isfinite(new) and old != 0:
        return (new - old) / old * 100
    return np.nan


# ============================================================
# LOAD STRAVA DATA
# ============================================================

def _load_strava_activity(activity):

    print(f"Loading activity: {activity.name} (ID {activity.id})")

    streams = client.get_activity_streams(
        activity.id,
        types=[
            "time", "distance", "velocity_smooth", "moving",
            "heartrate", "cadence", "altitude", "latlng",
        ],
        resolution="high",
    )

    required_streams = ["time", "distance", "velocity_smooth", "latlng"]
    missing = [s for s in required_streams if s not in streams]
    if missing:
        raise ValueError(f"Missing required Strava streams: {missing}")

    time_s = np.asarray(streams["time"].data, dtype=float)
    distance_m = np.asarray(streams["distance"].data, dtype=float)
    speed_ms = np.asarray(streams["velocity_smooth"].data, dtype=float)

    def optional(name):
        if name in streams:
            return np.asarray(streams[name].data, dtype=float)
        return np.full_like(time_s, np.nan, dtype=float)

    heart_rate = optional("heartrate")
    cadence = optional("cadence")
    altitude = optional("altitude")

    # Strava's moving stream marks each sample as moving or stopped.
    # Older activities may not provide it, so only then fall back to speed.
    if "moving" in streams:
        moving = np.asarray(streams["moving"].data, dtype=bool)
    else:
        moving = speed_ms > 0

    latlng = np.asarray(streams["latlng"].data, dtype=float)
    latitude = latlng[:, 0]
    longitude = latlng[:, 1]

    # Truncate to shortest stream.
    arrays = [
        time_s, distance_m, speed_ms, heart_rate, cadence,
        altitude, moving, latitude, longitude,
    ]
    n = min(len(a) for a in arrays)
    (
        time_s, distance_m, speed_ms, heart_rate, cadence,
        altitude, moving, latitude, longitude,
    ) = [a[:n] for a in arrays]

    # Unit conversion.
    time_min = time_s / 60.0
    distance_km = distance_m / 1000.0
    speed_kmh = speed_ms * 3.6

    pace_min_km = np.full_like(speed_ms, np.nan, dtype=float)
    positive_speed = speed_ms > 0
    pace_min_km[positive_speed] = 1000.0 / speed_ms[positive_speed] / 60.0

    # HR, cadence and altitude may stay NaN; they are ignored later.
    valid = (
        np.isfinite(time_min)
        & np.isfinite(distance_km)
        & np.isfinite(speed_ms)
        & np.isfinite(latitude)
        & np.isfinite(longitude)
    )
    order = np.argsort(time_min[valid])

    def clean(a):
        return a[valid][order]

    time_min = clean(time_min)
    distance_km = clean(distance_km)
    speed_kmh = clean(speed_kmh)
    pace_min_km = clean(pace_min_km)
    heart_rate = clean(heart_rate)
    cadence = clean(cadence)
    altitude = clean(altitude)
    moving = clean(moving)
    latitude = clean(latitude)
    longitude = clean(longitude)

    # Pauses never affect the pace trace, its range, or interval metrics.
    moving_pace = np.where(moving, pace_min_km, np.nan)
    pace_mean_global = np.nanmean(moving_pace)
    pace_std_global = np.nanstd(moving_pace)

    pace_plot = moving_pace.copy()
    pace_plot[
        (pace_plot < pace_mean_global - 3 * pace_std_global)
        | (pace_plot > pace_mean_global + 3 * pace_std_global)
    ] = np.nan

    # Use precisely the same moving, non-outlier samples for all metrics.
    analysis_mask = np.isfinite(pace_plot)

    return {
        "activity": activity,
        "time_min": time_min,
        "distance_km": distance_km,
        "speed_kmh": speed_kmh,
        "pace_min_km": pace_min_km,
        "pace_plot": pace_plot,
        "moving": moving,
        "analysis_mask": analysis_mask,
        "heart_rate": heart_rate,
        "cadence": cadence,
        "altitude": altitude,
        "latitude": latitude,
        "longitude": longitude,
    }


# ============================================================
# INTERVAL ANALYSIS
# ============================================================

def _analyze_interval(data, start, end):

    if end <= start:
        return None

    time_min = data["time_min"]
    time_mask = (time_min >= start) & (time_min <= end)
    mask = time_mask & data["analysis_mask"]

    if np.sum(mask) < 2:
        return None

    time_deltas = np.diff(time_min, prepend=time_min[0])
    duration = float(np.sum(time_deltas[mask]))

    mean_hr = _safe_nanmean(data["heart_rate"][mask])
    mean_cadence = _safe_nanmean(data["cadence"][mask])
    mean_speed = _safe_nanmean(data["speed_kmh"][mask])
    mean_altitude = _safe_nanmean(data["altitude"][mask])

    # Pace from mean speed rather than averaging instantaneous pace.
    if np.isfinite(mean_speed) and mean_speed > 0:
        mean_pace = 60.0 / mean_speed
    else:
        mean_pace = np.nan

    # Efficiency factor = speed / HR.
    if np.isfinite(mean_speed) and np.isfinite(mean_hr) and mean_hr > 0:
        efficiency_factor = mean_speed / mean_hr
    else:
        efficiency_factor = np.nan

    distance_deltas = np.diff(
        data["distance_km"], prepend=data["distance_km"][0]
    )
    interval_distance = float(np.sum(distance_deltas[mask]))

    return {
        "start": start,
        "end": end,
        "duration": duration,
        "hr": mean_hr,
        "cadence": mean_cadence,
        "speed": mean_speed,
        "pace": mean_pace,
        "altitude": mean_altitude,
        "ef": efficiency_factor,
        "distance": interval_distance,
        "mask": mask,
        "time_mask": time_mask,
    }


def _intervals_are_valid(i1_start, i1_end, i2_start, i2_end):
    """Interval 1 must end before interval 2 starts."""
    return i1_start < i1_end < i2_start < i2_end


def _calculate_comparison_metrics(interval_1, interval_2):

    if interval_1 is None or interval_2 is None:
        return {
            "hr_change": np.nan,
            "speed_change": np.nan,
            "pace_change": np.nan,
            "decoupling": np.nan,
        }

    # Decoupling = (EF1 - EF2) / EF1 * 100
    ef_change = _percent_change(interval_1["ef"], interval_2["ef"])

    return {
        "hr_change": _percent_change(interval_1["hr"], interval_2["hr"]),
        "speed_change": _percent_change(
            interval_1["speed"], interval_2["speed"]
        ),
        "pace_change": _percent_change(
            interval_1["pace"], interval_2["pace"]
        ),
        "decoupling": -ef_change if np.isfinite(ef_change) else np.nan,
    }


# ============================================================
# SANITIZE FIGUREWIDGET RELAYOUT MESSAGES
#
# Some Plotly front ends send browser-only "*._derived" keys
# during pan/zoom that older Python Plotly versions reject.
# ============================================================

def _strip_derived_relayout_keys(figure_widget):

    original_handler = figure_widget._handler_js2py_relayout

    def handler(change):
        relayout_msg = change["new"]
        if relayout_msg:
            relayout_data = relayout_msg.get("relayout_data") or {}
            for key in [
                k for k in relayout_data
                if k == "_derived" or k.endswith("._derived")
            ]:
                relayout_data.pop(key)
        original_handler(change)

    figure_widget._handler_js2py_relayout = handler


# ============================================================
# TIME-SERIES GRAPH
# ============================================================

def _axis_font(size_key):
    return dict(
        family=FONT_FAMILY,
        size=STYLE[size_key],
        color=WHITE,
    )


def _pace_axis_range(pace, low_pct=1, high_pct=97, pad=0.08):
    """
    Pace axis limits from the running part only. Short restarts and
    near-stops (e.g. 80 min/km) fall outside the percentiles and are
    simply drawn off-axis. Returned slow-to-fast, so faster is up.
    """
    pace = pace[np.isfinite(pace)]
    if pace.size == 0:
        return None
    fast, slow = np.nanpercentile(pace, [low_pct, high_pct])
    margin = max((slow - fast) * pad, 0.05)
    return [slow + margin, fast - margin]


def _create_time_plot(data, interval_1, interval_2, plot_width, plot_height):

    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
    )

    series = [
        ("pace_plot", "Pace: %{y:.2f} min/km"),
        ("heart_rate", "HR: %{y:.0f} bpm"),
        ("altitude", "Elevation: %{y:.0f} m"),
    ]

    for row, (key, hover) in enumerate(series, start=1):
        fig.add_trace(
            go.Scatter(
                x=data["time_min"],
                y=data[key],
                mode="lines",
                line=dict(width=1.5, color=WHITE),
                hovertemplate=f"{hover}<extra></extra>",
            ),
            row=row,
            col=1,
        )

    # Interval regions and dashed orange boundaries on all rows.
    annotations = []

    for interval in [interval_1, interval_2]:

        if interval is None:
            continue

        fig.add_vrect(
            x0=interval["start"],
            x1=interval["end"],
            fillcolor=ORANGE,
            opacity=0.12,
            line_width=0,
            row="all",
            col=1,
        )

        for x in [interval["start"], interval["end"]]:
            fig.add_vline(
                x=x,
                line_dash="dash",
                line_width=1.5,
                line_color=ORANGE,
                opacity=0.8,
                row="all",
                col=1,
            )


    # Horizontal y-axis labels: Plotly cannot rotate axis titles, so the
    # labels are annotations left-aligned at the figure's left edge.
    y_labels = [
        ("y domain", "Pace<br>(min/km)"),
        ("y2 domain", "HR<br>(bpm)"),
        ("y3 domain", "Elevation<br>(m)"),
    ]

    for yref, text in y_labels:
        annotations.append(
            dict(
                x=0,
                y=0.5,
                xref="paper",
                yref=yref,
                xanchor="left",
                yanchor="middle",
                xshift=-MARGIN_LEFT,
                align="left",
                text=text,
                showarrow=False,
                font=_axis_font("label_fontsize"),
            )
        )

    axis_style = dict(
        gridcolor=GRID_COLOR,
        gridwidth=0.5,
        zeroline=False,
        tickfont=_axis_font("small_fontsize"),
        title_font=_axis_font("label_fontsize"),
    )

    fig.update_yaxes(**axis_style)
    fig.update_yaxes(range=_pace_axis_range(data["pace_plot"]), row=1, col=1)
    fig.update_xaxes(**axis_style)
    fig.update_xaxes(title_text="Time (min)", row=3, col=1)

    fig.update_layout(
        autosize=plot_width is None,
        width=plot_width,
        height=plot_height,
        paper_bgcolor=BACKGROUND,
        plot_bgcolor=BACKGROUND,
        font=dict(family=FONT_FAMILY, color=WHITE),
        hoverlabel=dict(font=dict(family=FONT_FAMILY)),
        hovermode="x unified",
        margin=dict(l=MARGIN_LEFT, r=MARGIN_RIGHT, t=10, b=45),
        showlegend=False,
        annotations=annotations,
        uirevision="aerobic-decoupling-plot",
    )

    return fig


# ============================================================
# GPS MAP
# ============================================================

def _create_map(data, interval_1, interval_2, map_width, map_height):

    fig = go.Figure()

    fig.add_trace(
        go.Scattermap(
            lat=data["latitude"],
            lon=data["longitude"],
            mode="lines",
            line=dict(width=3, color=WHITE),
            hoverinfo="skip",
        )
    )

    def add_interval(interval, label):

        if interval is None:
            mask = np.zeros(len(data["time_min"]), dtype=bool)
            endpoint_indices = [None, None]
        else:
            mask = interval["mask"]
            time_indices = np.where(interval["time_mask"])[0]
            endpoint_indices = (
                [time_indices[0], time_indices[-1]]
                if len(time_indices) >= 2
                else [None, None]
            )

        # Interval 2 gets a black underlay so it stays distinct on top
        # of interval 1 whenever their routes overlap.
        if label == "2":
            fig.add_trace(
                go.Scattermap(
                    lat=data["latitude"][mask],
                    lon=data["longitude"][mask],
                    mode="lines",
                    line=dict(width=10, color="black"),
                    hoverinfo="skip",
                )
            )

        fig.add_trace(
            go.Scattermap(
                lat=data["latitude"][mask],
                lon=data["longitude"][mask],
                mode="lines",
                line=dict(width=6, color=ORANGE),
                hovertext=[
                    (
                        f"Interval {label}"
                        f"<br>Time: {t:.1f} min"
                        f"<br>Distance: {d:.2f} km"
                        f"<br>Pace: {_format_pace(p)} /km"
                        f"<br>HR: {_safe_value(hr, 0)} bpm"
                        f"<br>Cadence: {_safe_value(cad, 0)} spm"
                        f"<br>Elevation: {_safe_value(alt, 0)} m"
                    )
                    for t, d, p, hr, cad, alt in zip(
                        data["time_min"][mask],
                        data["distance_km"][mask],
                        data["pace_min_km"][mask],
                        data["heart_rate"][mask],
                        data["cadence"][mask],
                        data["altitude"][mask],
                    )
                ],
                hoverinfo="text",
            )
        )

        # Black circle with orange ring and white number at both ends.
        for endpoint_index in endpoint_indices:

            has_point = endpoint_index is not None
            lat = [data["latitude"][endpoint_index]] if has_point else []
            lon = [data["longitude"][endpoint_index]] if has_point else []

            fig.add_trace(
                go.Scattermap(
                    lat=lat, lon=lon, mode="markers",
                    marker=dict(size=36, color=ORANGE),
                    hoverinfo="skip",
                )
            )

            fig.add_trace(
                go.Scattermap(
                    lat=lat, lon=lon, mode="markers+text",
                    marker=dict(size=30, color="black"),
                    text=[label] if has_point else [],
                    # Map text is drawn with the map style's own glyph
                    # fonts, so only fonts like "Open Sans" work here.
                    textfont=dict(
                        family="Open Sans Bold", size=18, color=WHITE,
                    ),
                    textposition="middle center",
                    hoverinfo="skip",
                )
            )

    add_interval(interval_1, "1")
    add_interval(interval_2, "2")

    fig.update_layout(
        autosize=map_width is None,
        width=map_width,
        height=map_height,
        uirevision="aerobic-decoupling-map",
        paper_bgcolor=BACKGROUND,
        font=dict(family=FONT_FAMILY, color=WHITE),
        hoverlabel=dict(font=dict(family=FONT_FAMILY)),
        margin=dict(l=MARGIN_LEFT, r=MARGIN_RIGHT, t=10, b=10),
        map=dict(
            style="carto-darkmatter",
            center=dict(
                lat=float(np.nanmean(data["latitude"])),
                lon=float(np.nanmean(data["longitude"])),
            ),
            zoom=_route_zoom(
                data["latitude"],
                data["longitude"],
                (map_width or 700) - MARGIN_LEFT - MARGIN_RIGHT,
                map_height - 20,
            ),
        ),
        showlegend=False,
    )

    return fig


# ============================================================
# STATISTICS HTML
# ============================================================

def _create_stats_html(interval_1, interval_2, intervals_valid):

    wrapper = f"""
        font-family:{FONT_FAMILY};
        color:{WHITE};
        background:{BACKGROUND};
        padding:5px;
        width:100%;
        box-sizing:border-box;
    """

    if not intervals_valid:
        return f"""
        <div style="{wrapper}">
            <div style="color:{ORANGE};font-size:16px;font-weight:bold;">
                Invalid interval order
            </div>
            <div style="color:#AAAAAA;font-size:12px;margin-top:8px;">
                Interval 1 must end before interval 2 starts.
            </div>
        </div>
        """

    if interval_1 is None or interval_2 is None:
        return f"""
        <div style="{wrapper}color:{ORANGE};">
            Select two valid intervals.
        </div>
        """

    metrics = _calculate_comparison_metrics(interval_1, interval_2)
    decoupling = metrics["decoupling"]

    if not np.isfinite(decoupling):
        decoupling_label = "Unavailable"
    elif decoupling < 0:
        decoupling_label = "Negative"
    elif decoupling < 3:
        decoupling_label = "Very low"
    elif decoupling < 5:
        decoupling_label = "Low"
    elif decoupling < 10:
        decoupling_label = "Moderate"
    else:
        decoupling_label = "High"

    def interval_card(label, interval):
        return f"""
        <div style="
            flex:1;
            border:1px solid {ORANGE};
            border-radius:6px;
            padding:9px;
            background:{BACKGROUND};
        ">
            <div style="color:{ORANGE};font-weight:bold;font-size:13px;">
                INTERVAL {label}
            </div>
            <div style="color:#999999;font-size:11px;margin-top:4px;">
                {interval["start"]:.1f} → {interval["end"]:.1f} min
            </div>
            <div style="font-size:16px;font-weight:bold;margin-top:3px;">
                {_format_duration(interval["duration"])}
            </div>
        </div>
        """

    rows = [
        ("Distance", "distance", lambda v: f"{_safe_value(v, 2)} km"),
        ("Mean HR", "hr", lambda v: f"{_safe_value(v, 1)} bpm"),
        ("Mean cadence", "cadence", lambda v: f"{_safe_value(v, 1)} spm"),
        ("Mean speed", "speed", lambda v: f"{_safe_value(v, 2)} km/h"),
        ("Mean pace", "pace", lambda v: f"{_format_pace(v)} /km"),
        ("Mean elevation", "altitude", lambda v: f"{_safe_value(v, 0)} m"),
    ]

    cell = "padding:5px 3px;"

    table_rows = "".join(
        f"""
        <tr>
            <td style="{cell}">{name}</td>
            <td style="{cell}">{fmt(interval_1[key])}</td>
            <td style="{cell}">{fmt(interval_2[key])}</td>
        </tr>
        """
        for name, key, fmt in rows
    )

    changes = "".join(
        f"""
        <div style="display:flex;justify-content:space-between;padding:4px 0;">
            <span>{name}</span>
            <b>{_format_change(metrics[key])}</b>
        </div>
        """
        for name, key in [
            ("HR", "hr_change"),
            ("Speed", "speed_change"),
            ("Pace", "pace_change"),
        ]
    )

    return f"""
    <div style="{wrapper}">

        <div style="font-size:18px;font-weight:bold;margin-bottom:12px;">
            Interval analysis
        </div>

        <div style="display:flex;gap:8px;margin-bottom:12px;">
            {interval_card("1", interval_1)}
            {interval_card("2", interval_2)}
        </div>

        <table style="border-collapse:collapse;width:100%;font-size:11px;">
            <tr style="border-bottom:1px solid {BORDER};color:#888888;">
                <td style="padding:6px 3px;">Metric</td>
                <td style="padding:6px 3px;color:{ORANGE};font-weight:bold;">I1</td>
                <td style="padding:6px 3px;color:{ORANGE};font-weight:bold;">I2</td>
            </tr>
            {table_rows}
            <tr style="border-top:1px solid {BORDER};font-weight:bold;">
                <td style="padding:6px 3px;">Efficiency factor</td>
                <td style="padding:6px 3px;">{_safe_value(interval_1["ef"], 5)}</td>
                <td style="padding:6px 3px;">{_safe_value(interval_2["ef"], 5)}</td>
            </tr>
        </table>

        <div style="
            border-top:1px solid {BORDER};
            margin-top:12px;
            padding-top:10px;
        ">
            <div style="
                color:#888888;
                font-size:10px;
                text-transform:uppercase;
                letter-spacing:0.5px;
                margin-bottom:6px;
            ">
                Changes
            </div>
            {changes}
        </div>

        <div style="
            margin-top:13px;
            padding:13px;
            border:1px solid {ORANGE};
            border-radius:6px;
            background:{BACKGROUND};
        ">
            <div style="
                color:#888888;
                font-size:10px;
                text-transform:uppercase;
                letter-spacing:0.5px;
            ">
                Aerobic decoupling
            </div>
            <div style="
                margin-top:3px;
                font-size:26px;
                font-weight:bold;
                color:{ORANGE};
            ">
                {_format_change(decoupling)}
            </div>
            <div style="margin-top:2px;color:#888888;font-size:11px;">
                {decoupling_label}
            </div>
        </div>

    </div>
    """


# ============================================================
# DARK DASHBOARD CSS
# ============================================================

def _create_dark_css():

    return HTML(
        f"""
        <style>

        /* Black background and borders for every container. */
        .aero-dashboard,
        .aero-dashboard .widget-box,
        .aero-dashboard .widget-vbox,
        .aero-dashboard .widget-hbox,
        .aero-dashboard .widget-html,
        .aero-dashboard .widget-html-content,
        .aero-dashboard .widget-output,
        .aero-dashboard .widget-slider,
        .aero-dashboard .slider-container {{
            background-color: {BACKGROUND} !important;
            border-color: {BORDER} !important;
            color: {WHITE} !important;
        }}

        /* Widths include padding, so the panels add up exactly. */
        .aero-dashboard,
        .aero-dashboard * {{
            box-sizing: border-box !important;
        }}

        /* Never grow past the requested dashboard width. */
        .aero-dashboard {{
            flex: 0 0 auto !important;
            overflow: hidden !important;
        }}

        /* Notebook only: the cell output area around the dashboard is
           not part of the figure; paint it black to match. */
        .jp-OutputArea-output:has(.aero-dashboard),
        .cell-output-ipywidget-background:has(.aero-dashboard),
        .output_subarea:has(.aero-dashboard) {{
            background-color: {BACKGROUND} !important;
        }}

        /* Statistics panel: fixed width, never scrollbars. */
        .aero-dashboard .stats-panel,
        .aero-dashboard .stats-panel .widget-html,
        .aero-dashboard .stats-panel .widget-html-content {{
            overflow: hidden !important;
            margin: 0 !important;
            padding: 0 !important;
            height: auto !important;
            max-width: 100% !important;
        }}

        .aero-dashboard,
        .aero-dashboard .widget-label,
        .aero-dashboard .widget-readout {{
            font-family: {FONT_FAMILY} !important;
        }}

        /* ---------------------------------------------------------
           Single 4-handle interval slider.

           Two FloatRangeSliders are stacked in the same grid cell.
           The top slider (interval 2) ignores pointer events except
           on its own handles, so all four handles stay draggable.
           --------------------------------------------------------- */

        .aero-dashboard .interval-track {{
            display: grid !important;
            padding: 14px 0 !important;
            box-sizing: border-box !important;
        }}

        .aero-dashboard .interval-track > .widget-slider {{
            grid-row: 1;
            grid-column: 1;
            width: 100% !important;
            max-width: 100% !important;
            margin: 0 !important;
        }}

        .aero-dashboard .interval-track .widget-label {{
            display: none !important;
        }}

        .aero-dashboard .interval-track .slider-container {{
            margin: 0 !important;
            padding: 0 !important;
            width: 100% !important;
        }}

        .aero-dashboard .interval-2-range {{
            pointer-events: none !important;
        }}

        .aero-dashboard .interval-2-range .noUi-target,
        .aero-dashboard .interval-2-range .noUi-base,
        .aero-dashboard .interval-2-range .noUi-connects {{
            background: transparent !important;
            border: none !important;
            box-shadow: none !important;
        }}

        .aero-dashboard .interval-1-range .noUi-target {{
            background: {DARK_GRAY} !important;
            border: none !important;
            box-shadow: none !important;
        }}

        .aero-dashboard .interval-2-range .noUi-handle {{
            pointer-events: auto !important;
        }}

        .aero-dashboard .interval-1-range .noUi-connect,
        .aero-dashboard .interval-2-range .noUi-connect {{
            background: {ORANGE} !important;
        }}

        .aero-dashboard .interval-track .noUi-handle {{
            width: 30px !important;
            height: 30px !important;
            top: -13px !important;
            background: #000000 !important;
            border: 3px solid {ORANGE} !important;
            border-radius: 50% !important;
            box-shadow: none !important;
            cursor: grab !important;
        }}

        .aero-dashboard .interval-track .noUi-handle::before {{
            position: absolute !important;
            top: 0 !important;
            left: 0 !important;
            width: 100% !important;
            height: 100% !important;
            display: flex !important;
            align-items: center !important;
            justify-content: center !important;
            background: none !important;
            color: {WHITE} !important;
            font-family: {FONT_FAMILY} !important;
            font-size: 15px !important;
            font-weight: bold !important;
        }}

        .aero-dashboard .interval-1-range .noUi-handle::before {{
            content: "1" !important;
        }}

        .aero-dashboard .interval-2-range .noUi-handle::before {{
            content: "2" !important;
        }}

        .aero-dashboard .interval-track .noUi-handle::after {{
            display: none !important;
        }}

        </style>
        """
    )


# ============================================================
# LAUNCH FUNCTION
# ============================================================

def launch_aerobic_decoupling_analyzer(
    activity,
    dashboard_width="1300px",
    right_panel_width="360px",
    plot_height=650,
    map_height=550,
):
    """
    Launch the interactive aerobic decoupling dashboard.

    Parameters
    ----------
    activity:
        Strava activity object.

    dashboard_width:
        Total width in pixels, e.g. "1300px" or 1300. The right panel
        keeps `right_panel_width`; the graphs and map take the rest.
        Non-pixel values such as "100%" fall back to Plotly autosizing.

    right_panel_width:
        Width of the statistics panel.

    plot_height, map_height:
        Heights of the graphs and the map in pixels.
    """

    data = _load_strava_activity(activity)

    # With pixel widths the graphs and map get an exact width: the total
    # minus the right panel. Otherwise they autosize to their container.
    total_px = _px(dashboard_width)
    right_px = _px(right_panel_width)
    left_width = (
        total_px - right_px
        if total_px is not None and right_px is not None
        else None
    )

    min_time = float(np.nanmin(data["time_min"]))
    max_time = float(np.nanmax(data["time_min"]))
    span = max_time - min_time

    # ========================================================
    # INTERVAL SLIDER (two range sliders drawn as one track)
    # ========================================================

    def make_range_slider(start_fraction, end_fraction, css_class):
        slider = widgets.FloatRangeSlider(
            value=(
                min_time + span * start_fraction,
                min_time + span * end_fraction,
            ),
            min=min_time,
            max=max_time,
            step=0.1,
            description="",
            readout=False,
            continuous_update=True,
            style={"description_width": "0px"},
            layout=widgets.Layout(width="100%"),
        )
        slider.add_class(css_class)
        return slider

    i1_range = make_range_slider(0.20, 0.40, "interval-1-range")
    i2_range = make_range_slider(0.60, 0.80, "interval-2-range")

    interval_track = widgets.Box(
        [i1_range, i2_range],
        layout=widgets.Layout(flex="1 1 0", min_width="0"),
    )
    interval_track.add_class("interval-track")

    # The label fills the same left margin as the graph y-labels, so the
    # slider track lines up with the time axis below it.
    interval_label = widgets.HTML(
        value=f"""
        <div style="
            font-family:{FONT_FAMILY};
            font-size:{STYLE["label_fontsize"]}px;
            color:{WHITE};
            line-height:1.25;
        ">
            Interval<br>selection
        </div>
        """,
        layout=widgets.Layout(
            width=f"{MARGIN_LEFT}px",
            min_width=f"{MARGIN_LEFT}px",
            margin="0",
        ),
    )

    interval_row = widgets.HBox(
        [interval_label, interval_track],
        layout=widgets.Layout(
            width="100%",
            align_items="center",
            padding=f"12px {MARGIN_RIGHT}px 12px 0",
        ),
    )

    # ========================================================
    # FIGURES (kept alive so zoom/pan survive slider updates)
    # ========================================================

    initial_i1 = dict(zip(("start", "end"), i1_range.value))
    initial_i2 = dict(zip(("start", "end"), i2_range.value))

    plot_widget = go.FigureWidget(
        _create_time_plot(data, initial_i1, initial_i2, left_width, plot_height)
    )
    map_widget = go.FigureWidget(
        _create_map(data, None, None, left_width, map_height)
    )

    _strip_derived_relayout_keys(plot_widget)
    _strip_derived_relayout_keys(map_widget)

    # FigureWidget.layout is Plotly's layout, so size via wrapper boxes.
    plot_box = widgets.Box(
        [plot_widget], layout=widgets.Layout(width="100%")
    )
    map_box = widgets.Box(
        [map_widget], layout=widgets.Layout(width="100%")
    )

    stats_panel = widgets.HTML(
        layout=widgets.Layout(width="100%", margin="0", overflow="hidden")
    )

    # ========================================================
    # LAYOUT
    # ========================================================

    header = widgets.HTML(
        value=f"""
        <div style="
            background:{BACKGROUND};
            color:{WHITE};
            font-family:{FONT_FAMILY};
            font-size:21px;
            font-weight:bold;
            padding:4px 0 12px 0;
            white-space:nowrap;
            overflow:hidden;
            text-overflow:ellipsis;
        ">
            Aerobic Decoupling in:
            <span style="color:{ORANGE};">{activity.name}</span>
        </div>
        """,
        layout=widgets.Layout(width="100%"),
    )

    left_panel = widgets.VBox(
        [interval_row, plot_box, map_box],
        layout=widgets.Layout(
            width=f"{left_width}px" if left_width else "auto",
            flex="0 0 auto" if left_width else "1 1 0",
            min_width="0",
        ),
    )

    right_panel = widgets.VBox(
        [stats_panel],
        layout=widgets.Layout(
            width=right_panel_width,
            min_width=right_panel_width,
            max_width=right_panel_width,
            flex="0 0 auto",
            overflow="hidden",
        ),
    )
    right_panel.add_class("stats-panel")

    body = widgets.HBox(
        [left_panel, right_panel],
        layout=widgets.Layout(
            width="100%",
            flex_flow="row nowrap",
            align_items="flex-start",
        ),
    )

    dashboard = widgets.VBox(
        [header, body],
        layout=widgets.Layout(width=dashboard_width),
    )
    dashboard.add_class("aero-dashboard")

    # ========================================================
    # UPDATE
    # ========================================================

    def update(change=None):

        start_1, end_1 = i1_range.value
        start_2, end_2 = i2_range.value

        intervals_valid = _intervals_are_valid(
            start_1, end_1, start_2, end_2
        )

        if intervals_valid:
            interval_1 = _analyze_interval(data, start_1, end_1)
            interval_2 = _analyze_interval(data, start_2, end_2)
        else:
            interval_1 = None
            interval_2 = None

        # The graph always shows the selected regions.
        updated_plot = _create_time_plot(
            data,
            {"start": start_1, "end": end_1},
            {"start": start_2, "end": end_2},
            left_width,
            plot_height,
        )

        with plot_widget.batch_update():
            plot_widget.layout.shapes = updated_plot.layout.shapes
            plot_widget.layout.annotations = updated_plot.layout.annotations

        # The map only highlights validly ordered intervals.
        updated_map = _create_map(
            data, interval_1, interval_2, left_width, map_height
        )

        with map_widget.batch_update():
            for current_trace, updated_trace in zip(
                map_widget.data, updated_map.data
            ):
                current_trace.update(updated_trace)

        stats_panel.value = _create_stats_html(
            interval_1, interval_2, intervals_valid
        )

    for slider in [i1_range, i2_range]:
        slider.observe(update, names="value")

    display(_create_dark_css())
    display(dashboard)

    update()

    return {
        "activity": activity,
        "data": data,
        "dashboard": dashboard,
        "i1_range": i1_range,
        "i2_range": i2_range,
    }

In [39]:
# ============================================================
# INTERACTIVE STRAVA AEROBIC DECOUPLING ANALYZER
#
#   ┌───────────────────────────────────────────────────────┐
#   │ Aerobic Decoupling in: <session name>                 │
#   ├──────────────────────────────┬────────────────────────┤
#   │ Interval slider (4 handles)  │                        │
#   │ Pace / HR / Elevation        │  Statistics            │
#   │ GPS map                      │  (fixed width)         │
#   └──────────────────────────────┴────────────────────────┘
#
# Change `dashboard_width` to resize everything: the right
# panel keeps its width, the graphs and map take the rest.
# ============================================================

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import display, HTML

from strava_data.visualization import COLORS, STYLE


# ============================================================
# COLORS / GLOBAL STYLE
# ============================================================

BACKGROUND = "#000000"
BORDER = "#000000"
ORANGE = COLORS["main"]
WHITE = COLORS["neutral"]
GRAY = COLORS["dark"]
DARK_GRAY = COLORS["darker"]
GRID_COLOR = COLORS["darker"]

# One font stack for HTML, widgets and Plotly, so the browser picks the
# same fallback everywhere when the primary font is not installed.
FONT_FAMILY = f"{STYLE['font_family']}, Arial, sans-serif"

# Shared horizontal margins so the map lines up with the graph area.
MARGIN_LEFT = 95
MARGIN_RIGHT = 20


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def _format_pace(pace):
    """Convert decimal minutes/km to mm:ss/km."""
    if not np.isfinite(pace):
        return "—"
    total_seconds = pace * 60.0
    minutes = int(total_seconds // 60)
    seconds = int(round(total_seconds % 60))
    if seconds == 60:
        minutes += 1
        seconds = 0
    return f"{minutes}:{seconds:02d}"


def _format_duration(minutes):
    """Convert decimal minutes to mm:ss or hh:mm:ss."""
    if not np.isfinite(minutes):
        return "—"
    total_seconds = int(round(minutes * 60))
    hours = total_seconds // 3600
    mins = (total_seconds % 3600) // 60
    secs = total_seconds % 60
    if hours > 0:
        return f"{hours}:{mins:02d}:{secs:02d}"
    return f"{mins}:{secs:02d}"


def _format_change(value):
    """Format percentage change."""
    if not np.isfinite(value):
        return "—"
    return f"{value:+.2f}%"


def _safe_nanmean(values):
    """np.nanmean without warnings if all values are NaN."""
    values = np.asarray(values)
    if not np.any(np.isfinite(values)):
        return np.nan
    return np.nanmean(values)


def _safe_value(value, decimals=1):
    """Format numerical values while handling NaN."""
    if not np.isfinite(value):
        return "—"
    return f"{value:.{decimals}f}"


def _px(value):
    """Return a pixel width for ints or "1300px" strings, else None."""
    if isinstance(value, (int, float)):
        return int(value)
    if isinstance(value, str) and value.strip().endswith("px"):
        return int(float(value.strip()[:-2]))
    return None


def _route_zoom(latitude, longitude, width_px, height_px, padding=0.9):
    """
    Largest web-mercator zoom level at which the whole route fits in
    a map of width_px x height_px (MapLibre uses 512 px tiles).
    """
    lat = np.radians(latitude[np.isfinite(latitude)])
    lon = longitude[np.isfinite(longitude)]

    lon_span = max(float(np.ptp(lon)), 1e-6)
    merc_y = np.log(np.tan(np.pi / 4 + lat / 2))
    y_span = max(float(np.ptp(merc_y)), 1e-6)

    zoom_lon = np.log2(360.0 * width_px * padding / (512.0 * lon_span))
    zoom_lat = np.log2(2 * np.pi * height_px * padding / (512.0 * y_span))

    return float(np.clip(min(zoom_lon, zoom_lat), 1, 18))


# ============================================================
# NORMALIZED GRADED PACE (NGP)
#
# 1) Grade adjustment: speed on a slope is converted to the
#    flat-ground speed with the same energy cost, using the
#    running energy-cost model of Minetti et al. (2002):
#
#       C(i) = 155.4 i^5 - 30.4 i^4 - 43.3 i^3 + 46.3 i^2
#              + 19.5 i + 3.6          (J/kg/m, i = grade)
#
#       graded speed = speed * C(i) / C(0)
#
# 2) Normalization, as for Normalized Power: 30 s rolling mean of
#    graded speed, raised to the 4th power, averaged, 4th root.
#
# TrainingPeaks' exact Normalized Graded Pace model is proprietary; this is the
# published physiology it is based on, so values can differ a
# little from TrainingPeaks.
# ============================================================

def _running_cost(grade):
    i = np.clip(grade, -0.45, 0.45)
    return (
        155.4 * i**5 - 30.4 * i**4 - 43.3 * i**3
        + 46.3 * i**2 + 19.5 * i + 3.6
    )


def _grade_from_altitude(distance_m, altitude, window_m=20.0):
    """Fallback grade (fraction) from altitude over a ±10 m window."""
    grade = np.zeros_like(distance_m, dtype=float)
    ok = np.isfinite(altitude) & np.isfinite(distance_m)
    if ok.sum() < 2:
        return grade
    d, idx = np.unique(distance_m[ok], return_index=True)
    alt = altitude[ok][idx]
    if d.size < 2:
        return grade
    half = window_m / 2
    lo = np.clip(distance_m - half, d[0], d[-1])
    hi = np.clip(distance_m + half, d[0], d[-1])
    span = hi - lo
    rise = np.interp(hi, d, alt) - np.interp(lo, d, alt)
    with np.errstate(invalid="ignore", divide="ignore"):
        grade = np.where(span > 1.0, rise / span, 0.0)
    return np.nan_to_num(grade)


def _normalized_speed(speed, time_s, window_s=30.0):
    """
    4th-power normalization of a speed series over a 30 s rolling
    window. `time_s` is cumulative *moving* time, so stops never
    enter a window.
    """
    if speed.size < 2:
        return np.nan

    csum = np.concatenate([[0.0], np.cumsum(speed)])
    left = np.searchsorted(time_s, time_s - window_s, side="right")
    counts = np.arange(1, speed.size + 1) - left
    rolling = (csum[1:] - csum[left]) / counts

    # Like NP, skip the first window when the interval is long enough.
    if time_s[-1] - time_s[0] > 2 * window_s:
        rolling = rolling[time_s - time_s[0] >= window_s]

    return float(np.mean(rolling**4) ** 0.25)


def _percent_change(old, new):
    if np.isfinite(old) and np.isfinite(new) and old != 0:
        return (new - old) / old * 100
    return np.nan


# ============================================================
# LOAD STRAVA DATA
# ============================================================

def _load_strava_activity(activity):

    print(f"Loading activity: {activity.name} (ID {activity.id})")

    streams = client.get_activity_streams(
        activity.id,
        types=[
            "time", "distance", "velocity_smooth", "moving",
            "heartrate", "cadence", "altitude", "latlng",
            "grade_smooth",
        ],
        resolution="high",
    )

    required_streams = ["time", "distance", "velocity_smooth", "latlng"]
    missing = [s for s in required_streams if s not in streams]
    if missing:
        raise ValueError(f"Missing required Strava streams: {missing}")

    time_s = np.asarray(streams["time"].data, dtype=float)
    distance_m = np.asarray(streams["distance"].data, dtype=float)
    speed_ms = np.asarray(streams["velocity_smooth"].data, dtype=float)

    def optional(name):
        if name in streams:
            return np.asarray(streams[name].data, dtype=float)
        return np.full_like(time_s, np.nan, dtype=float)

    heart_rate = optional("heartrate")
    cadence = optional("cadence")
    altitude = optional("altitude")
    # Strava's smoothed grade in percent; NaN if not provided.
    grade_pct = optional("grade_smooth")

    # Strava's moving stream marks each sample as moving or stopped.
    # Older activities may not provide it, so only then fall back to speed.
    if "moving" in streams:
        moving = np.asarray(streams["moving"].data, dtype=bool)
    else:
        moving = speed_ms > 0

    latlng = np.asarray(streams["latlng"].data, dtype=float)
    latitude = latlng[:, 0]
    longitude = latlng[:, 1]

    # Truncate to shortest stream.
    arrays = [
        time_s, distance_m, speed_ms, heart_rate, cadence,
        altitude, grade_pct, moving, latitude, longitude,
    ]
    n = min(len(a) for a in arrays)
    (
        time_s, distance_m, speed_ms, heart_rate, cadence,
        altitude, grade_pct, moving, latitude, longitude,
    ) = [a[:n] for a in arrays]

    # Unit conversion.
    time_min = time_s / 60.0
    distance_km = distance_m / 1000.0
    speed_kmh = speed_ms * 3.6

    pace_min_km = np.full_like(speed_ms, np.nan, dtype=float)
    positive_speed = speed_ms > 0
    pace_min_km[positive_speed] = 1000.0 / speed_ms[positive_speed] / 60.0

    # HR, cadence and altitude may stay NaN; they are ignored later.
    valid = (
        np.isfinite(time_min)
        & np.isfinite(distance_km)
        & np.isfinite(speed_ms)
        & np.isfinite(latitude)
        & np.isfinite(longitude)
    )
    order = np.argsort(time_min[valid])

    def clean(a):
        return a[valid][order]

    time_min = clean(time_min)
    distance_km = clean(distance_km)
    speed_kmh = clean(speed_kmh)
    pace_min_km = clean(pace_min_km)
    heart_rate = clean(heart_rate)
    cadence = clean(cadence)
    altitude = clean(altitude)
    grade_pct = clean(grade_pct)
    moving = clean(moving)
    latitude = clean(latitude)
    longitude = clean(longitude)

    # Pauses never affect the pace trace, its range, or interval metrics.
    moving_pace = np.where(moving, pace_min_km, np.nan)
    pace_mean_global = np.nanmean(moving_pace)
    pace_std_global = np.nanstd(moving_pace)

    pace_plot = moving_pace.copy()
    pace_plot[
        (pace_plot < pace_mean_global - 3 * pace_std_global)
        | (pace_plot > pace_mean_global + 3 * pace_std_global)
    ] = np.nan

    # Use precisely the same moving, non-outlier samples for all metrics.
    analysis_mask = np.isfinite(pace_plot)

    # Grade: Strava's grade_smooth when available, else from altitude.
    if np.any(np.isfinite(grade_pct)):
        grade = np.nan_to_num(grade_pct / 100.0)
    else:
        grade = _grade_from_altitude(distance_km * 1000.0, altitude)

    graded_speed_kmh = speed_kmh * _running_cost(grade) / _running_cost(0.0)

    return {
        "activity": activity,
        "time_min": time_min,
        "distance_km": distance_km,
        "speed_kmh": speed_kmh,
        "pace_min_km": pace_min_km,
        "pace_plot": pace_plot,
        "moving": moving,
        "analysis_mask": analysis_mask,
        "heart_rate": heart_rate,
        "cadence": cadence,
        "altitude": altitude,
        "grade": grade,
        "graded_speed_kmh": graded_speed_kmh,
        "latitude": latitude,
        "longitude": longitude,
    }


# ============================================================
# INTERVAL ANALYSIS
# ============================================================

def _analyze_interval(data, start, end):

    if end <= start:
        return None

    time_min = data["time_min"]
    time_mask = (time_min >= start) & (time_min <= end)
    mask = time_mask & data["analysis_mask"]

    if np.sum(mask) < 2:
        return None

    time_deltas = np.diff(time_min, prepend=time_min[0])
    duration = float(np.sum(time_deltas[mask]))

    mean_hr = _safe_nanmean(data["heart_rate"][mask])
    mean_cadence = _safe_nanmean(data["cadence"][mask])
    mean_speed = _safe_nanmean(data["speed_kmh"][mask])
    mean_altitude = _safe_nanmean(data["altitude"][mask])

    # Pace from mean speed rather than averaging instantaneous pace.
    if np.isfinite(mean_speed) and mean_speed > 0:
        mean_pace = 60.0 / mean_speed
    else:
        mean_pace = np.nan

    # Normalized graded speed over moving, non-outlier samples only.
    # Moving time is rebuilt from sample gaps (capped at 5 s, so a paused
    # recording does not create a long gap inside the rolling window).
    moving_dt_s = np.minimum(time_deltas[mask] * 60.0, 5.0)
    moving_time_s = np.cumsum(moving_dt_s)
    ngp_speed = _normalized_speed(
        data["graded_speed_kmh"][mask], moving_time_s
    )

    if np.isfinite(ngp_speed) and ngp_speed > 0:
        ngp = 60.0 / ngp_speed
    else:
        ngp = np.nan

    # Efficiency factor = normalized graded speed / HR (as TrainingPeaks).
    if np.isfinite(ngp_speed) and np.isfinite(mean_hr) and mean_hr > 0:
        efficiency_factor = ngp_speed / mean_hr
    else:
        efficiency_factor = np.nan

    distance_deltas = np.diff(
        data["distance_km"], prepend=data["distance_km"][0]
    )
    interval_distance = float(np.sum(distance_deltas[mask]))

    return {
        "start": start,
        "end": end,
        "duration": duration,
        "hr": mean_hr,
        "cadence": mean_cadence,
        "speed": mean_speed,
        "pace": mean_pace,
        "ngp_speed": ngp_speed,
        "ngp": ngp,
        "altitude": mean_altitude,
        "ef": efficiency_factor,
        "distance": interval_distance,
        "mask": mask,
        "time_mask": time_mask,
    }


def _intervals_are_valid(i1_start, i1_end, i2_start, i2_end):
    """Interval 1 must end before interval 2 starts."""
    return i1_start < i1_end < i2_start < i2_end


def _calculate_comparison_metrics(interval_1, interval_2):

    if interval_1 is None or interval_2 is None:
        return {
            "hr_change": np.nan,
            "pace_change": np.nan,
            "ngp_change": np.nan,
            "decoupling": np.nan,
        }

    # Decoupling = (EF1 - EF2) / EF1 * 100
    ef_change = _percent_change(interval_1["ef"], interval_2["ef"])

    return {
        "hr_change": _percent_change(interval_1["hr"], interval_2["hr"]),
        "pace_change": _percent_change(
            interval_1["pace"], interval_2["pace"]
        ),
        "ngp_change": _percent_change(
            interval_1["ngp"], interval_2["ngp"]
        ),
        "decoupling": -ef_change if np.isfinite(ef_change) else np.nan,
    }


# ============================================================
# SANITIZE FIGUREWIDGET RELAYOUT MESSAGES
#
# Some Plotly front ends send browser-only "*._derived" keys
# during pan/zoom that older Python Plotly versions reject.
# ============================================================

def _strip_derived_relayout_keys(figure_widget):

    original_handler = figure_widget._handler_js2py_relayout

    def handler(change):
        relayout_msg = change["new"]
        if relayout_msg:
            relayout_data = relayout_msg.get("relayout_data") or {}
            for key in [
                k for k in relayout_data
                if k == "_derived" or k.endswith("._derived")
            ]:
                relayout_data.pop(key)
        original_handler(change)

    figure_widget._handler_js2py_relayout = handler


# ============================================================
# TIME-SERIES GRAPH
# ============================================================

def _axis_font(size_key):
    return dict(
        family=FONT_FAMILY,
        size=STYLE[size_key],
        color=WHITE,
    )


def _pace_axis_range(pace, low_pct=1, high_pct=97, pad=0.08):
    """
    Pace axis limits from the running part only. Short restarts and
    near-stops (e.g. 80 min/km) fall outside the percentiles and are
    simply drawn off-axis. Returned slow-to-fast, so faster is up.
    """
    pace = pace[np.isfinite(pace)]
    if pace.size == 0:
        return None
    fast, slow = np.nanpercentile(pace, [low_pct, high_pct])
    margin = max((slow - fast) * pad, 0.05)
    return [slow + margin, fast - margin]


def _create_time_plot(data, interval_1, interval_2, plot_width, plot_height):

    fig = make_subplots(
        rows=3,
        cols=1,
        shared_xaxes=True,
        vertical_spacing=0.05,
    )

    series = [
        ("pace_plot", "Pace: %{y:.2f} min/km"),
        ("heart_rate", "HR: %{y:.0f} bpm"),
        ("altitude", "Elevation: %{y:.0f} m"),
    ]

    for row, (key, hover) in enumerate(series, start=1):
        fig.add_trace(
            go.Scatter(
                x=data["time_min"],
                y=data[key],
                mode="lines",
                line=dict(width=1.5, color=WHITE),
                hovertemplate=f"{hover}<extra></extra>",
            ),
            row=row,
            col=1,
        )

    # Interval regions and dashed orange boundaries on all rows.
    annotations = []

    for interval in [interval_1, interval_2]:

        if interval is None:
            continue

        fig.add_vrect(
            x0=interval["start"],
            x1=interval["end"],
            fillcolor=ORANGE,
            opacity=0.12,
            line_width=0,
            row="all",
            col=1,
        )

        for x in [interval["start"], interval["end"]]:
            fig.add_vline(
                x=x,
                line_dash="dash",
                line_width=1.5,
                line_color=ORANGE,
                opacity=0.8,
                row="all",
                col=1,
            )


    # Horizontal y-axis labels: Plotly cannot rotate axis titles, so the
    # labels are annotations left-aligned at the figure's left edge.
    y_labels = [
        ("y domain", "Pace<br>(min/km)"),
        ("y2 domain", "HR<br>(bpm)"),
        ("y3 domain", "Elevation<br>(m)"),
    ]

    for yref, text in y_labels:
        annotations.append(
            dict(
                x=0,
                y=0.5,
                xref="paper",
                yref=yref,
                xanchor="left",
                yanchor="middle",
                xshift=-MARGIN_LEFT,
                align="left",
                text=text,
                showarrow=False,
                font=_axis_font("label_fontsize"),
            )
        )

    axis_style = dict(
        gridcolor=GRID_COLOR,
        gridwidth=0.5,
        zeroline=False,
        tickfont=_axis_font("small_fontsize"),
        title_font=_axis_font("label_fontsize"),
    )

    fig.update_yaxes(**axis_style)
    fig.update_yaxes(range=_pace_axis_range(data["pace_plot"]), row=1, col=1)
    fig.update_xaxes(**axis_style)
    fig.update_xaxes(title_text="Time (min)", row=3, col=1)

    fig.update_layout(
        autosize=plot_width is None,
        width=plot_width,
        height=plot_height,
        paper_bgcolor=BACKGROUND,
        plot_bgcolor=BACKGROUND,
        font=dict(family=FONT_FAMILY, color=WHITE),
        hoverlabel=dict(font=dict(family=FONT_FAMILY)),
        hovermode="x unified",
        margin=dict(l=MARGIN_LEFT, r=MARGIN_RIGHT, t=10, b=45),
        showlegend=False,
        annotations=annotations,
        uirevision="aerobic-decoupling-plot",
    )

    return fig


# ============================================================
# GPS MAP
# ============================================================

def _create_map(data, interval_1, interval_2, map_width, map_height):

    fig = go.Figure()

    fig.add_trace(
        go.Scattermap(
            lat=data["latitude"],
            lon=data["longitude"],
            mode="lines",
            line=dict(width=3, color=WHITE),
            hoverinfo="skip",
        )
    )

    def add_interval(interval, label):

        if interval is None:
            mask = np.zeros(len(data["time_min"]), dtype=bool)
            endpoint_indices = [None, None]
        else:
            mask = interval["mask"]
            time_indices = np.where(interval["time_mask"])[0]
            endpoint_indices = (
                [time_indices[0], time_indices[-1]]
                if len(time_indices) >= 2
                else [None, None]
            )

        # Interval 2 gets a black underlay so it stays distinct on top
        # of interval 1 whenever their routes overlap.
        if label == "2":
            fig.add_trace(
                go.Scattermap(
                    lat=data["latitude"][mask],
                    lon=data["longitude"][mask],
                    mode="lines",
                    line=dict(width=10, color="black"),
                    hoverinfo="skip",
                )
            )

        fig.add_trace(
            go.Scattermap(
                lat=data["latitude"][mask],
                lon=data["longitude"][mask],
                mode="lines",
                line=dict(width=6, color=ORANGE),
                hovertext=[
                    (
                        f"Interval {label}"
                        f"<br>Time: {t:.1f} min"
                        f"<br>Distance: {d:.2f} km"
                        f"<br>Pace: {_format_pace(p)} /km"
                        f"<br>HR: {_safe_value(hr, 0)} bpm"
                        f"<br>Cadence: {_safe_value(cad, 0)} spm"
                        f"<br>Elevation: {_safe_value(alt, 0)} m"
                    )
                    for t, d, p, hr, cad, alt in zip(
                        data["time_min"][mask],
                        data["distance_km"][mask],
                        data["pace_min_km"][mask],
                        data["heart_rate"][mask],
                        data["cadence"][mask],
                        data["altitude"][mask],
                    )
                ],
                hoverinfo="text",
            )
        )

        # Black circle with orange ring and white number at both ends.
        for endpoint_index in endpoint_indices:

            has_point = endpoint_index is not None
            lat = [data["latitude"][endpoint_index]] if has_point else []
            lon = [data["longitude"][endpoint_index]] if has_point else []

            fig.add_trace(
                go.Scattermap(
                    lat=lat, lon=lon, mode="markers",
                    marker=dict(size=36, color=ORANGE),
                    hoverinfo="skip",
                )
            )

            fig.add_trace(
                go.Scattermap(
                    lat=lat, lon=lon, mode="markers+text",
                    marker=dict(size=30, color="black"),
                    text=[label] if has_point else [],
                    # Map text is drawn with the map style's own glyph
                    # fonts, so only fonts like "Open Sans" work here.
                    textfont=dict(
                        family="Open Sans Bold", size=18, color=WHITE,
                    ),
                    textposition="middle center",
                    hoverinfo="skip",
                )
            )

    add_interval(interval_1, "1")
    add_interval(interval_2, "2")

    fig.update_layout(
        autosize=map_width is None,
        width=map_width,
        height=map_height,
        uirevision="aerobic-decoupling-map",
        paper_bgcolor=BACKGROUND,
        font=dict(family=FONT_FAMILY, color=WHITE),
        hoverlabel=dict(font=dict(family=FONT_FAMILY)),
        margin=dict(l=MARGIN_LEFT, r=MARGIN_RIGHT, t=10, b=10),
        map=dict(
            style="carto-darkmatter",
            center=dict(
                lat=float(np.nanmean(data["latitude"])),
                lon=float(np.nanmean(data["longitude"])),
            ),
            zoom=_route_zoom(
                data["latitude"],
                data["longitude"],
                (map_width or 700) - MARGIN_LEFT - MARGIN_RIGHT,
                map_height - 20,
            ),
        ),
        showlegend=False,
    )

    return fig


# ============================================================
# STATISTICS HTML
# ============================================================

def _create_stats_html(interval_1, interval_2, intervals_valid):

    wrapper = f"""
        font-family:{FONT_FAMILY};
        color:{WHITE};
        background:{BACKGROUND};
        padding:5px;
        width:100%;
        box-sizing:border-box;
    """

    if not intervals_valid:
        return f"""
        <div style="{wrapper}">
            <div style="color:{ORANGE};font-size:16px;font-weight:bold;">
                Invalid interval order
            </div>
            <div style="color:#AAAAAA;font-size:12px;margin-top:8px;">
                Interval 1 must end before interval 2 starts.
            </div>
        </div>
        """

    if interval_1 is None or interval_2 is None:
        return f"""
        <div style="{wrapper}color:{ORANGE};">
            Select two valid intervals.
        </div>
        """

    metrics = _calculate_comparison_metrics(interval_1, interval_2)
    decoupling = metrics["decoupling"]

    if not np.isfinite(decoupling):
        decoupling_label = "Unavailable"
    elif decoupling < 0:
        decoupling_label = "Negative"
    elif decoupling < 3:
        decoupling_label = "Very low"
    elif decoupling < 5:
        decoupling_label = "Low"
    elif decoupling < 10:
        decoupling_label = "Moderate"
    else:
        decoupling_label = "High"

    def metric(name, value, unit=""):
        unit_html = (
            f'<span style="font-size:12px;color:{WHITE};font-weight:normal;'
            f'margin-left:3px;">{unit}</span>'
            if unit and value != "—"
            else ""
        )
        return f"""
            <div style="margin-top:10px;">
                <div style="color:{GRAY};font-size:11px;line-height:1.2;">
                    {name}
                </div>
                <div style="
                    color:{WHITE};
                    font-size:22px;
                    font-weight:bold;
                    line-height:1.2;
                    white-space:nowrap;
                ">
                    {value}{unit_html}
                </div>
            </div>
        """

    def interval_card(label, interval):
        return f"""
        <div style="
            flex:1;
            min-width:0;
            border:1px solid {ORANGE};
            border-radius:6px;
            padding:10px 12px 12px 12px;
            background:{BACKGROUND};
        ">
            <div style="color:{ORANGE};font-weight:bold;font-size:13px;">
                INTERVAL {label}
            </div>
            <div style="color:{GRAY};font-size:11px;margin-top:2px;">
                {interval["start"]:.1f} → {interval["end"]:.1f} min
            </div>
            {metric("Time", _format_duration(interval["duration"]))}
            {metric("Distance", _safe_value(interval["distance"], 2), "km")}
            {metric("Pace", _format_pace(interval["pace"]), "/km")}
            {metric("Normalized Graded Pace", _format_pace(interval["ngp"]), "/km")}
            {metric("Heart rate", _safe_value(interval["hr"], 0), "bpm")}
            {metric("Cadence", _safe_value(interval["cadence"], 0), "spm")}
            {metric("Elevation", _safe_value(interval["altitude"], 0), "m")}
            {metric("Efficiency factor", _safe_value(interval["ef"], 4))}
        </div>
        """

    changes = "".join(
        f"""
        <div style="
            display:flex;
            justify-content:space-between;
            align-items:baseline;
            padding:4px 0;
        ">
            <span style="color:{GRAY};font-size:12px;">{name}</span>
            <span style="color:{WHITE};font-size:18px;font-weight:bold;">
                {_format_change(metrics[key])}
            </span>
        </div>
        """
        for name, key in [
            ("Heart rate", "hr_change"),
            ("Pace", "pace_change"),
            ("Normalized Graded Pace", "ngp_change"),
        ]
    )

    return f"""
    <div style="{wrapper}">

        <div style="font-size:18px;font-weight:bold;margin-bottom:12px;">
            Interval analysis
        </div>

        <div style="display:flex;gap:8px;">
            {interval_card("1", interval_1)}
            {interval_card("2", interval_2)}
        </div>

        <div style="
            margin-top:8px;
            padding:10px 12px;
            border:1px solid {BORDER};
            border-radius:6px;
            background:{BACKGROUND};
        ">
            <div style="
                color:#888888;
                font-size:13px;
                font-weight:bold;
                margin-bottom:4px;
            ">
                CHANGES
            </div>
            {changes}
        </div>


        <div style="
            margin-top:8px;
            padding:10px 12px;
            border:1px solid {ORANGE};
            border-radius:6px;
            background:{BACKGROUND};
        ">
            <div style="
                color:#888888;
                font-size:13px;
                font-weight:bold;
            ">
                AEROBIC DECOUPLING
            </div>
            <div style="
                margin-top:3px;
                font-size:26px;
                font-weight:bold;
                color:{ORANGE};
            ">
                {_format_change(decoupling)}
            </div>
            <div style="margin-top:2px;color:#888888;font-size:11px;">
                {decoupling_label}
            </div>
        </div>

    </div>
    """


# ============================================================
# DARK DASHBOARD CSS
# ============================================================

def _create_dark_css():

    return HTML(
        f"""
        <style>

        /* Black background and borders for every container. */
        .aero-dashboard,
        .aero-dashboard .widget-box,
        .aero-dashboard .widget-vbox,
        .aero-dashboard .widget-hbox,
        .aero-dashboard .widget-html,
        .aero-dashboard .widget-html-content,
        .aero-dashboard .widget-output,
        .aero-dashboard .widget-slider,
        .aero-dashboard .slider-container {{
            background-color: {BACKGROUND} !important;
            border-color: {BORDER} !important;
            color: {WHITE} !important;
        }}

        /* Widths include padding, so the panels add up exactly. */
        .aero-dashboard,
        .aero-dashboard * {{
            box-sizing: border-box !important;
        }}

        /* Never grow past the requested dashboard width. */
        .aero-dashboard {{
            flex: 0 0 auto !important;
            overflow: hidden !important;
        }}

        /* Notebook only: the cell output area around the dashboard is
           not part of the figure; paint it black to match. */
        .jp-OutputArea-output:has(.aero-dashboard),
        .cell-output-ipywidget-background:has(.aero-dashboard),
        .output_subarea:has(.aero-dashboard) {{
            background-color: {BACKGROUND} !important;
        }}

        /* Statistics panel: fixed width, never scrollbars. */
        .aero-dashboard .stats-panel,
        .aero-dashboard .stats-panel .widget-html,
        .aero-dashboard .stats-panel .widget-html-content {{
            overflow: hidden !important;
            margin: 0 !important;
            padding: 0 !important;
            height: auto !important;
            max-width: 100% !important;
        }}

        .aero-dashboard,
        .aero-dashboard .widget-label,
        .aero-dashboard .widget-readout {{
            font-family: {FONT_FAMILY} !important;
        }}

        /* ---------------------------------------------------------
           Single 4-handle interval slider.

           Two FloatRangeSliders are stacked in the same grid cell.
           The top slider (interval 2) ignores pointer events except
           on its own handles, so all four handles stay draggable.
           --------------------------------------------------------- */

        .aero-dashboard .interval-track {{
            display: grid !important;
            padding: 14px 0 !important;
            box-sizing: border-box !important;
        }}

        .aero-dashboard .interval-track > .widget-slider {{
            grid-row: 1;
            grid-column: 1;
            width: 100% !important;
            max-width: 100% !important;
            margin: 0 !important;
        }}

        .aero-dashboard .interval-track .widget-label {{
            display: none !important;
        }}

        .aero-dashboard .interval-track .slider-container {{
            margin: 0 !important;
            padding: 0 !important;
            width: 100% !important;
        }}

        .aero-dashboard .interval-2-range {{
            pointer-events: none !important;
        }}

        .aero-dashboard .interval-2-range .noUi-target,
        .aero-dashboard .interval-2-range .noUi-base,
        .aero-dashboard .interval-2-range .noUi-connects {{
            background: transparent !important;
            border: none !important;
            box-shadow: none !important;
        }}

        .aero-dashboard .interval-1-range .noUi-target {{
            background: {DARK_GRAY} !important;
            border: none !important;
            box-shadow: none !important;
        }}

        .aero-dashboard .interval-2-range .noUi-handle {{
            pointer-events: auto !important;
        }}

        .aero-dashboard .interval-1-range .noUi-connect,
        .aero-dashboard .interval-2-range .noUi-connect {{
            background: {ORANGE} !important;
        }}

        .aero-dashboard .interval-track .noUi-handle {{
            width: 30px !important;
            height: 30px !important;
            top: -13px !important;
            background: #000000 !important;
            border: 3px solid {ORANGE} !important;
            border-radius: 50% !important;
            box-shadow: none !important;
            cursor: grab !important;
        }}

        .aero-dashboard .interval-track .noUi-handle::before {{
            position: absolute !important;
            top: 0 !important;
            left: 0 !important;
            width: 100% !important;
            height: 100% !important;
            display: flex !important;
            align-items: center !important;
            justify-content: center !important;
            background: none !important;
            color: {WHITE} !important;
            font-family: {FONT_FAMILY} !important;
            font-size: 15px !important;
            font-weight: bold !important;
        }}

        .aero-dashboard .interval-1-range .noUi-handle::before {{
            content: "1" !important;
        }}

        .aero-dashboard .interval-2-range .noUi-handle::before {{
            content: "2" !important;
        }}

        .aero-dashboard .interval-track .noUi-handle::after {{
            display: none !important;
        }}

        </style>
        """
    )


# ============================================================
# LAUNCH FUNCTION
# ============================================================

def launch_aerobic_decoupling_analyzer(
    activity,
    dashboard_width="1300px",
    right_panel_width="360px",
    plot_height=650,
    map_height=550,
):
    """
    Launch the interactive aerobic decoupling dashboard.

    Parameters
    ----------
    activity:
        Strava activity object.

    dashboard_width:
        Total width in pixels, e.g. "1300px" or 1300. The right panel
        keeps `right_panel_width`; the graphs and map take the rest.
        Non-pixel values such as "100%" fall back to Plotly autosizing.

    right_panel_width:
        Width of the statistics panel.

    plot_height, map_height:
        Heights of the graphs and the map in pixels.
    """

    data = _load_strava_activity(activity)

    # With pixel widths the graphs and map get an exact width: the total
    # minus the right panel. Otherwise they autosize to their container.
    total_px = _px(dashboard_width)
    right_px = _px(right_panel_width)
    left_width = (
        total_px - right_px
        if total_px is not None and right_px is not None
        else None
    )

    min_time = float(np.nanmin(data["time_min"]))
    max_time = float(np.nanmax(data["time_min"]))
    span = max_time - min_time

    # ========================================================
    # INTERVAL SLIDER (two range sliders drawn as one track)
    # ========================================================

    def make_range_slider(start_fraction, end_fraction, css_class):
        slider = widgets.FloatRangeSlider(
            value=(
                min_time + span * start_fraction,
                min_time + span * end_fraction,
            ),
            min=min_time,
            max=max_time,
            step=0.1,
            description="",
            readout=False,
            continuous_update=True,
            style={"description_width": "0px"},
            layout=widgets.Layout(width="100%"),
        )
        slider.add_class(css_class)
        return slider

    i1_range = make_range_slider(0.20, 0.40, "interval-1-range")
    i2_range = make_range_slider(0.60, 0.80, "interval-2-range")

    interval_track = widgets.Box(
        [i1_range, i2_range],
        layout=widgets.Layout(flex="1 1 0", min_width="0"),
    )
    interval_track.add_class("interval-track")

    # The label fills the same left margin as the graph y-labels, so the
    # slider track lines up with the time axis below it.
    interval_label = widgets.HTML(
        value=f"""
        <div style="
            font-family:{FONT_FAMILY};
            font-size:{STYLE["label_fontsize"]}px;
            color:{WHITE};
            line-height:1.25;
        ">
            Interval<br>selection
        </div>
        """,
        layout=widgets.Layout(
            width=f"{MARGIN_LEFT}px",
            min_width=f"{MARGIN_LEFT}px",
            margin="0",
        ),
    )

    interval_row = widgets.HBox(
        [interval_label, interval_track],
        layout=widgets.Layout(
            width="100%",
            align_items="center",
            padding=f"12px {MARGIN_RIGHT}px 12px 0",
        ),
    )

    # ========================================================
    # FIGURES (kept alive so zoom/pan survive slider updates)
    # ========================================================

    initial_i1 = dict(zip(("start", "end"), i1_range.value))
    initial_i2 = dict(zip(("start", "end"), i2_range.value))

    plot_widget = go.FigureWidget(
        _create_time_plot(data, initial_i1, initial_i2, left_width, plot_height)
    )
    map_widget = go.FigureWidget(
        _create_map(data, None, None, left_width, map_height)
    )

    _strip_derived_relayout_keys(plot_widget)
    _strip_derived_relayout_keys(map_widget)

    # FigureWidget.layout is Plotly's layout, so size via wrapper boxes.
    plot_box = widgets.Box(
        [plot_widget], layout=widgets.Layout(width="100%")
    )
    map_box = widgets.Box(
        [map_widget], layout=widgets.Layout(width="100%")
    )

    stats_panel = widgets.HTML(
        layout=widgets.Layout(width="100%", margin="0", overflow="hidden")
    )

    # ========================================================
    # LAYOUT
    # ========================================================

    header = widgets.HTML(
        value=f"""
        <div style="
            background:{BACKGROUND};
            color:{WHITE};
            font-family:{FONT_FAMILY};
            font-size:21px;
            font-weight:bold;
            padding:4px 0 12px 0;
            white-space:nowrap;
            overflow:hidden;
            text-overflow:ellipsis;
        ">
            Aerobic Decoupling in:
            <span style="color:{ORANGE};">{activity.name}</span>
        </div>
        """,
        layout=widgets.Layout(width="100%"),
    )

    left_panel = widgets.VBox(
        [interval_row, plot_box, map_box],
        layout=widgets.Layout(
            width=f"{left_width}px" if left_width else "auto",
            flex="0 0 auto" if left_width else "1 1 0",
            min_width="0",
        ),
    )

    right_panel = widgets.VBox(
        [stats_panel],
        layout=widgets.Layout(
            width=right_panel_width,
            min_width=right_panel_width,
            max_width=right_panel_width,
            flex="0 0 auto",
            overflow="hidden",
        ),
    )
    right_panel.add_class("stats-panel")

    body = widgets.HBox(
        [left_panel, right_panel],
        layout=widgets.Layout(
            width="100%",
            flex_flow="row nowrap",
            align_items="flex-start",
        ),
    )

    dashboard = widgets.VBox(
        [header, body],
        layout=widgets.Layout(width=dashboard_width),
    )
    dashboard.add_class("aero-dashboard")

    # ========================================================
    # UPDATE
    # ========================================================

    def update(change=None):

        start_1, end_1 = i1_range.value
        start_2, end_2 = i2_range.value

        intervals_valid = _intervals_are_valid(
            start_1, end_1, start_2, end_2
        )

        if intervals_valid:
            interval_1 = _analyze_interval(data, start_1, end_1)
            interval_2 = _analyze_interval(data, start_2, end_2)
        else:
            interval_1 = None
            interval_2 = None

        # The graph always shows the selected regions.
        updated_plot = _create_time_plot(
            data,
            {"start": start_1, "end": end_1},
            {"start": start_2, "end": end_2},
            left_width,
            plot_height,
        )

        with plot_widget.batch_update():
            plot_widget.layout.shapes = updated_plot.layout.shapes
            plot_widget.layout.annotations = updated_plot.layout.annotations

        # The map only highlights validly ordered intervals.
        updated_map = _create_map(
            data, interval_1, interval_2, left_width, map_height
        )

        with map_widget.batch_update():
            for current_trace, updated_trace in zip(
                map_widget.data, updated_map.data
            ):
                current_trace.update(updated_trace)

        stats_panel.value = _create_stats_html(
            interval_1, interval_2, intervals_valid
        )

    for slider in [i1_range, i2_range]:
        slider.observe(update, names="value")

    display(_create_dark_css())
    display(dashboard)

    update()

    return {
        "activity": activity,
        "data": data,
        "dashboard": dashboard,
        "i1_range": i1_range,
        "i2_range": i2_range,
    }

In [40]:
# ============================================================
# ACTIVITY SELECTION
# ============================================================

activity = activities[29] # 24


# ============================================================
# LAUNCH
# ============================================================

analyzer = launch_aerobic_decoupling_analyzer(
    activity,
    dashboard_width="1920px",
    right_panel_width="500px",
    plot_height=350,
    map_height=350,
)

Loading activity: Nachtloop 🌧️ (ID 19587873354)
